1. Can we turn the Maybe type constructor into a functor by defining:```fmap _ _ = Nothing```
which ignores both of its arguments? (Hint: Check the functor
laws.)

```haskell
fmap id Nothing = Nothing = id Nothing
fmap id (Just x) = Nothing /= Just x = id (Just x)
```

2. Prove functor laws for the reader functor. Hint: it’s really simple.

```haskell
fmap id f = id . g
          = g
          = id g
fmap (g . h) f = (g . h) . f
               = g . (h . f)
               = fmap g (h . f)
               = fmap g (fmap h f)
               = (fmap g . fmap h) f
```

3. Implement the reader functor in your second favorite language
(the first being Haskell, of course).


In [2]:
%%writefile reader.cpp
#include <iostream>

template <typename A, typename B, typename R>
struct reader {

    using AtoB = B(A);
    using RtoB = B(R);    
    using RtoA = A(R);

    auto fmap(RtoA f, AtoB g) {
        return [=] (R r) { return g(f(r)); };
    }

};

auto main () -> int{

    auto string_to_float = [](auto s) { return std::stof(s); };
    auto float_to_int    = [](auto f) { return static_cast<int>(f); };
    auto reader_fmap     = [](auto f, auto g) { return [f, g] (auto r) { return g(f(r)); }; };
    auto string_to_int2   = 
        reader<float, int, std::string>{}.fmap(string_to_float, float_to_int);

    auto string_to_int = reader_fmap(string_to_float, float_to_int);

    for (auto s : { "1.23", "42.42", "17.29"}) {
        std::cout << string_to_float(s) << std::endl;
        std::cout << string_to_int(s) << std::endl;
    }
}

Writing reader.cpp


In [5]:
!g++ -std=c++17 -o reader reader.cpp && ./reader && rm ./reader.cpp ./reader

1.23
1
42.42
42
17.29
17


4. Prove the functor laws for the list functor. Assume that the laws are true for the tail part of the list you’re applying it to (in other words, use induction).

```haskell
fmap _ Nil = Nil
fmap f (Cons x t) = Cons (f x) (fmap f t)

fmap id Nil = Nil = id Nil
fmap id (Cons x t) = Cons (id x) (fmap id t)
                   = Cons x (fmap id t)
                   = Cons x t
                   = id (Cons x t)

fmap (g . h) Nil = Nil = fmap g (fmap h Nil) = (fmap g . fmap h) Nil
fmap (g . h) (Cons x t) = Cons ((g . h) x) (fmap (g . h) t)
                        = Cons ((g . h) x) ((fmap g . fmap h) t)
                        = Cons (g (h x)) (fmap g (fmap h t))
                        = fmap g (Cons (h x) (fmap h t))
                        = fmap g (fmap h (Cons x t))
                        = (fmap g . fmap h) (Cons x t)
```